# 数据准备：编码转换、数据合并与时间修复（2017-2020）

本 Notebook 包含以下步骤：
1. 将 `gb18030` 原始文件转换为 `utf-8-sig`
2. 合并 2017-2018 与 2019-2020 借阅数据
3. 修复无空格时间格式并转换为 `datetime`
4. 生成三张标准表：用户信息表、图书信息表、借阅表


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base_raw = Path('../data/raw')
base_processed = Path('../data/processed')
base_processed.mkdir(parents=True, exist_ok=True)

src_2017_gb = base_raw / 'LENDHIST2017_2018_gb18030.csv'
src_2019_utf8 = base_raw / 'LENDHIST2019_2020_utf8.csv'
dst_2017_utf8 = base_raw / 'LENDHIST2017_2018_utf8.csv'

print('--- Step 1: 编码转换 (gb18030 -> utf-8-sig) ---')
df_2017 = pd.read_csv(src_2017_gb, encoding='gb18030', low_memory=False)
df_2017.to_csv(dst_2017_utf8, index=False, encoding='utf-8-sig')
print(f'已生成: {dst_2017_utf8}')
print(f'2017-2018 数据规模: {df_2017.shape}')

print('\n--- Step 2: 读取并合并两期数据 ---')
df_2017 = pd.read_csv(dst_2017_utf8, low_memory=False)
df_2019 = pd.read_csv(src_2019_utf8, low_memory=False)
raw = pd.concat([df_2017, df_2019], ignore_index=True)
raw = raw.replace('nan', np.nan)
print(f'合并后总记录: {raw.shape[0]}, 字段数: {raw.shape[1]}')

print('\n--- Step 3: 时间格式修复与转换 ---')
# 原始格式示例: 2019-11-2909:46:42（日期和时间中间缺空格）
for col in ['LEND_DATE', 'RET_DATE']:
    s = raw[col].astype(str)
    s = s.str.replace(r'^(\d{4}-\d{2}-\d{2})(\d{2}:\d{2}:\d{2})$', r'\1 \2', regex=True)
    s = s.replace({'nan': np.nan, 'NaT': np.nan, '': np.nan})
    raw[col] = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d %H:%M:%S')

print('LEND_DATE 解析失败条数:', int(raw['LEND_DATE'].isna().sum()))
ret_notna = raw['RET_DATE'].notna().sum()
print('RET_DATE 非空条数:', int(ret_notna))

print('\n--- Step 4: 生成三张标准表 ---')
# 用借阅时间排序后，维度表按最新记录保留
raw['_order_time'] = raw['LEND_DATE'].fillna(pd.Timestamp('1900-01-01'))
raw = raw.sort_values('_order_time')

# 4.1 用户信息表
user_cols = ['USERID', 'BIRTHYEAR', 'SEX', 'DEPT', 'OCCUPATION', 'CODE01', 'REDR_TYPE_NAME']
users = raw[user_cols + ['_order_time']].drop_duplicates(subset=['USERID'], keep='last').drop(columns=['_order_time'])

# 4.2 图书信息表
book_cols = ['BOOK_ID', 'TITLE', 'ABSTRACT', 'SUB', 'CALL_NO', 'AUTHOR', 'AU', 'PUBLISHER', 'ISBN', 'PUB_YEAR', 'DOC_TYPE_NAME']
books = raw[book_cols + ['_order_time']].drop_duplicates(subset=['BOOK_ID'], keep='last').drop(columns=['_order_time'])

# 4.3 借阅事实表（保留全部借阅记录）
borrow_cols = ['USERID', 'BOOK_ID', 'LEND_DATE', 'RET_DATE', 'RENEW_TIMES', 'LOCATION_NAME']
borrows = raw[borrow_cols].copy()
borrows.insert(0, 'BORROW_ID', range(1, len(borrows) + 1))

users_path = base_processed / 'users_2017_2020.csv'
books_path = base_processed / 'books_2017_2020.csv'
borrows_path = base_processed / 'borrows_2017_2020.csv'

users.to_csv(users_path, index=False, encoding='utf-8-sig')
books.to_csv(books_path, index=False, encoding='utf-8-sig')
borrows.to_csv(borrows_path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')

print(f'用户表: {users.shape} -> {users_path}')
print(f'图书表: {books.shape} -> {books_path}')
print(f'借阅表: {borrows.shape} -> {borrows_path}')

print('\n完成。')
